# Live GIS: HKU walkability from real OpenStreetMap data

<a href="https://colab.research.google.com/github/clementinec/ARCH7476/blob/gh-pages/starter_kits/gis-spatial-comparison/gis_live_osm_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook extends `gis_spatial_comparison_demo.py`. It uses the same three candidate sites near HKU, the same four metrics and the same scoring formula. The difference is where the layers come from:

| | `gis_spatial_comparison_demo.py` | this notebook |
|---|---|---|
| layers | hand-drawn GeoJSON, synthetic grocery points | **downloaded live** from OpenStreetMap |
| catchment | 300 m straight-line circle | circle **and** 300 m along the real walking network |
| noise | drawn corridor polygons | buffered major roads (road-class proxy) |
| extra | – | building coverage, detour ratio, "walkable share of the circle" |

**The question it answers:** the old preview asked *"does the 300 m catchment measure real access, or just distance on a map?"* Here we measure both and see whether the site ranking changes.

**Run order:** `Runtime → Run all`. The first live download takes about 20–60 s. If the OSM server is busy (common when a whole class runs at once), set `LIVE = False` in cell 2 to load the cached snapshot instead.

In [ ]:
# 1 · Setup. Colab already has geopandas; osmnx is the only extra piece.
import importlib.util, sys
if importlib.util.find_spec("osmnx") is None:
    %pip install -q osmnx folium mapclassify
import osmnx as ox, geopandas as gpd, pandas as pd, networkx as nx, numpy as np
import matplotlib.pyplot as plt, folium, json, datetime
from pathlib import Path
from shapely.geometry import shape, Point
print("osmnx", ox.__version__, "| geopandas", gpd.__version__)

## 2 · Parameters: the only cell you need to edit

- `CENTER` / `DIST_M`: the area to download. Point it at your own studio site (right-click in Google Maps to copy lat, lon).
- `SITES`: candidate sites as GeoJSON polygons. The three below are the same teaching polygons used by the existing demo. **To test your own site,** use `site_square(...)` (see the commented example).
- `WALK_RADIUS_M`: catchment distance, used for both the circle and the walking network.
- `WEIGHTS`: same as the existing demo. Changing them is itself a design argument, so write down why.

In [ ]:
LIVE = True                       # False → use the cached snapshot (no internet download)
CENTER = (22.2841, 114.1355)      # (lat, lon) — HKU Station area
DIST_M = 1000                     # download radius around CENTER, metres
WALK_RADIUS_M = 300               # catchment: 300 m ≈ 4 min walk on the flat (much longer uphill!)
CRS_WORK = "EPSG:2326"            # HK1980 Grid — metric, low distortion in Hong Kong
WEIGHTS = {"transit": 0.30, "grocery": 0.25, "green": 0.20, "noise": 0.25}

def site_square(site_id, lat, lon, label, half_m=25):
    # Quick square site polygon around a point, for testing your own location.
    dlat, dlon = half_m / 111_320, half_m / (111_320 * np.cos(np.radians(lat)))
    ring = [[lon-dlon, lat-dlat], [lon+dlon, lat-dlat], [lon+dlon, lat+dlat], [lon-dlon, lat+dlat], [lon-dlon, lat-dlat]]
    return {"type": "Feature", "properties": {"site_id": site_id, "current_use": label},
            "geometry": {"type": "Polygon", "coordinates": [ring]}}

SITES = {"type": "FeatureCollection", "features": [
    {"type": "Feature", "properties": {"site_id": "A", "current_use": "HKU station forecourt edge"},
     "geometry": {"type": "Polygon", "coordinates": [[[114.13520, 22.28420], [114.13570, 22.28420], [114.13570, 22.28455], [114.13520, 22.28455], [114.13520, 22.28420]]]}},
    {"type": "Feature", "properties": {"site_id": "B", "current_use": "main campus edge"},
     "geometry": {"type": "Polygon", "coordinates": [[[114.13655, 22.28275], [114.13705, 22.28275], [114.13705, 22.28315], [114.13655, 22.28315], [114.13655, 22.28275]]]}},
    {"type": "Feature", "properties": {"site_id": "C", "current_use": "Hill Road pocket"},
     "geometry": {"type": "Polygon", "coordinates": [[[114.13245, 22.28580], [114.13285, 22.28580], [114.13285, 22.28615], [114.13245, 22.28615], [114.13245, 22.28580]]]}},
    # site_square("D", 22.2830, 114.1400, "my studio site"),   # ← uncomment / edit to add your own
]}

## 3 · Download the layers from OpenStreetMap

Each layer is an **OSM tag query**. The tag list *is* your definition of "transit", "grocery", and so on, and it is the first thing to question during the audit.

| layer | OSM tags used | what it misses |
|---|---|---|
| walk network | `network_type="walk"` (footways, steps, streets) | many podium/mall interiors and some footbridges |
| transit | `highway=bus_stop`, `railway=station`, `public_transport=station` | minibus stops are patchy |
| grocery | `shop=supermarket/convenience/greengrocer` | shops inside malls, wet markets |
| green | `leisure=park/garden/playground`, `landuse=grass/recreation_ground` | private podium gardens |
| noise proxy | `highway=motorway/trunk/primary/secondary` buffered 25 m | actual measured dB |
| buildings | `building=*` | fairly complete on HK Island |

In [ ]:
KIT = Path.cwd()
CACHE = KIT / "data" / "osm_cache"
OUT = KIT / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
RAW = "https://raw.githubusercontent.com/clementinec/ARCH7476/gh-pages/starter_kits/gis-spatial-comparison/data/osm_cache/"

TAGS = {
    "transit":   {"highway": "bus_stop", "railway": "station", "public_transport": "station"},
    "grocery":   {"shop": ["supermarket", "convenience", "greengrocer"]},
    "green":     {"leisure": ["park", "garden", "playground"], "landuse": ["grass", "recreation_ground"]},
    "roads":     {"highway": ["motorway", "trunk", "primary", "secondary"]},
    "buildings": {"building": True},
}
LAYER_GEOMS = {"transit": ["Point", "Polygon", "MultiPolygon"], "grocery": ["Point", "Polygon", "MultiPolygon"],
               "green": ["Polygon", "MultiPolygon"], "roads": ["LineString", "MultiLineString"],
               "buildings": ["Polygon", "MultiPolygon"]}

def cache_src(name):
    # Local cache when running from the repo; the GitHub copy when running in Colab.
    local = CACHE / name
    return str(local) if local.exists() else RAW + name

def download():
    t0 = datetime.datetime.now()
    G = ox.graph_from_point(CENTER, dist=DIST_M, network_type="walk")
    layers = {}
    for key, tags in TAGS.items():
        f = ox.features_from_point(CENTER, tags, dist=DIST_M)
        f = f[f.geom_type.isin(LAYER_GEOMS[key])].reset_index()
        keep = [c for c in ["osmid", "id", "name", "shop", "highway", "railway", "leisure", "landuse", "building"] if c in f.columns]
        layers[key] = f[keep + ["geometry"]].copy()
    print(f"downloaded in {(datetime.datetime.now() - t0).seconds}s")
    return G, layers

def save_cache(G, layers):
    CACHE.mkdir(parents=True, exist_ok=True)
    ox.save_graphml(G, CACHE / "walk_network.graphml")
    for key, gdf in layers.items():
        gdf.astype({c: str for c in gdf.columns if c != "geometry"}).to_file(CACHE / f"{key}.geojson", driver="GeoJSON")
    (CACHE / "snapshot.json").write_text(json.dumps({"center": CENTER, "dist_m": DIST_M,
        "downloaded": datetime.date.today().isoformat(), "source": "OpenStreetMap contributors, ODbL"}, indent=2))

def load_cache():
    src = cache_src("walk_network.graphml")
    if src.startswith("http"):
        import urllib.request
        CACHE.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(src, CACHE / "walk_network.graphml")
        src = CACHE / "walk_network.graphml"
    G = ox.load_graphml(src)
    layers = {key: gpd.read_file(cache_src(f"{key}.geojson")) for key in TAGS}
    return G, layers

if LIVE:
    try:
        G, layers = download()
        SNAPSHOT = f"live OSM download, {datetime.date.today().isoformat()}"
        if CENTER == (22.2841, 114.1355) and DIST_M == 1000:
            save_cache(G, layers)    # refresh the shared HKU snapshot only for the default area
    except Exception as err:
        print("Live download failed:", repr(err)[:200], "\n→ falling back to cached snapshot")
        LIVE = False
if not LIVE:
    G, layers = load_cache()
    SNAPSHOT = "cached OSM snapshot"

print(f"source: {SNAPSHOT} · © OpenStreetMap contributors (ODbL)")
print(f"walk network: {len(G.nodes):,} nodes, {len(G.edges):,} edges")
for key, gdf in layers.items():
    print(f"{key:>10}: {len(gdf):,} features")

## 4 · Project everything to metres

OSM delivers WGS84 (degrees). Buffers and areas need metres, so we project to **EPSG:2326 (HK1980 Grid)**. The existing demo used EPSG:3857 (Web Mercator), which overstates distances by about 8% at 22°N. That doesn't matter for a sketch, but it's the kind of thing the audit should catch.

In [ ]:
Gp = ox.project_graph(G, to_crs=CRS_WORK)
nodes, edges = ox.graph_to_gdfs(Gp)
L = {key: gdf.to_crs(CRS_WORK) for key, gdf in layers.items()}
sites = gpd.GeoDataFrame.from_features(SITES["features"], crs="EPSG:4326").to_crs(CRS_WORK)
road_zone = L["roads"].buffer(25).union_all() if len(L["roads"]) else None
green_union = L["green"].union_all() if len(L["green"]) else None
bldg_union = L["buildings"].union_all() if len(L["buildings"]) else None
sites

## 5 · Two catchments per site

1. **Circle**: everything within 300 m in a straight line, the same as the existing demo.
2. **Walking-network isochrone**: start at the street node nearest the site, walk 300 m along real footpaths, streets and steps, then take the streets you reached plus a 25 m strip on each side.

On a flat grid the two are similar. On Mid-Levels slopes with retaining walls they are not.

In [ ]:
def isochrone(G_proj, nodes_gdf, edges_gdf, origin_pt, cutoff_m, strip_m=25):
    origin = ox.distance.nearest_nodes(G_proj, origin_pt.x, origin_pt.y)
    dist = nx.single_source_dijkstra_path_length(G_proj, origin, cutoff=cutoff_m, weight="length")
    reached = set(dist)
    e = edges_gdf.reset_index()
    e = e[e["u"].isin(reached) & e["v"].isin(reached)]
    poly = e.buffer(strip_m).union_all() if len(e) else origin_pt.buffer(strip_m)
    return poly, origin, dist

def network_nearest_m(G_proj, origin_dist, targets, origin_snap_m):
    # Shortest walking distance to the nearest target (snap each target to its nearest street node).
    if len(targets) == 0:
        return np.nan
    pts = targets.geometry.representative_point()
    tnodes = ox.distance.nearest_nodes(G_proj, pts.x.values, pts.y.values)
    best = np.inf
    for n, p in zip(tnodes, pts):
        if n in origin_dist:
            node_xy = Point(G_proj.nodes[n]["x"], G_proj.nodes[n]["y"])
            best = min(best, origin_dist[n] + node_xy.distance(p))
    return best + origin_snap_m if np.isfinite(best) else np.nan

def share_pct(union, zone):
    return 0.0 if union is None else 100 * union.intersection(zone).area / zone.area

def distance_score(distance_m, count, bonus):
    return min(100, max(0, 100 - distance_m / 3) + count * bonus)   # identical to existing demo

def score(nt, ct, ng, cg, ngr, gpct, npct):
    s_transit = distance_score(nt, ct, 8)
    s_grocery = distance_score(ng, cg, 10)
    s_green = min(100, max(0, 100 - ngr / 4) + gpct * 1.5)
    s_noise = max(0, 100 - npct * 7)
    w = WEIGHTS
    return w["transit"] * s_transit + w["grocery"] * s_grocery + w["green"] * s_green + w["noise"] * s_noise

rows, iso_geoms = [], []
for _, site in sites.iterrows():
    c = site.geometry.centroid
    circle = c.buffer(WALK_RADIUS_M)
    iso, origin, dist = isochrone(Gp, nodes, edges, c, WALK_RADIUS_M)
    snap_m = Point(Gp.nodes[origin]["x"], Gp.nodes[origin]["y"]).distance(c)
    dist_long = nx.single_source_dijkstra_path_length(Gp, origin, cutoff=3000, weight="length")
    r = {"site_id": site.site_id, "current_use": site.current_use}
    for tag, zone in [("circle", circle), ("walk", iso)]:
        if tag == "circle":
            nt = L["transit"].distance(c).min(); ng = L["grocery"].distance(c).min(); ngr = L["green"].distance(c).min()
        else:
            nt = network_nearest_m(Gp, dist_long, L["transit"], snap_m)
            ng = network_nearest_m(Gp, dist_long, L["grocery"], snap_m)
            ngr = network_nearest_m(Gp, dist_long, L["green"], snap_m)
        ct = int(L["transit"].intersects(zone).sum()); cg = int(L["grocery"].intersects(zone).sum())
        gpct = share_pct(green_union, zone); npct = share_pct(road_zone, zone)
        r.update({f"{tag}_nearest_transit_m": round(nt, 1), f"{tag}_transit_n": ct,
                  f"{tag}_nearest_grocery_m": round(ng, 1), f"{tag}_grocery_n": cg,
                  f"{tag}_nearest_green_m": round(ngr, 1), f"{tag}_green_pct": round(gpct, 1),
                  f"{tag}_noise_pct": round(npct, 1),
                  f"{tag}_score": round(score(nt, ct, ng, cg, ngr, gpct, npct), 1)})
    r["building_cover_pct"] = round(share_pct(bldg_union, circle), 1)
    r["walkable_share_of_circle_pct"] = round(100 * iso.intersection(circle).area / circle.area, 1)
    r["transit_detour_ratio"] = round(r["walk_nearest_transit_m"] / max(r["circle_nearest_transit_m"], 1), 2)
    rows.append(r)
    iso_geoms.append(iso)

scored = pd.DataFrame(rows)
scored["circle_rank"] = scored["circle_score"].rank(ascending=False, method="min").astype(int)
scored["walk_rank"] = scored["walk_score"].rank(ascending=False, method="min").astype(int)
isos = gpd.GeoDataFrame(scored[["site_id"]], geometry=iso_geoms, crs=CRS_WORK)
print("done:", len(scored), "sites scored twice (circle vs walking network)")

## 6 · Does the ranking change?

The key columns:
- `walkable_share_of_circle_pct`: how much of the 300 m circle you can reach within 300 m of actual walking. 100% would mean a perfect flat grid.
- `transit_detour_ratio`: walking distance ÷ straight-line distance to the nearest stop. Above about 1.5, the circle is misleading.
- `circle_rank` vs `walk_rank`: if these differ, the straight-line method would have picked a different site.

In [ ]:
cols = ["site_id", "current_use", "circle_score", "walk_score", "circle_rank", "walk_rank",
        "walkable_share_of_circle_pct", "transit_detour_ratio",
        "circle_nearest_transit_m", "walk_nearest_transit_m",
        "circle_nearest_grocery_m", "walk_nearest_grocery_m",
        "walk_transit_n", "walk_grocery_n", "walk_green_pct", "walk_noise_pct", "building_cover_pct"]
view = scored[cols].sort_values("walk_score", ascending=False)
changed = (scored["circle_rank"] != scored["walk_rank"]).any()
print("Ranking changes between circle and walking network:", "YES" if changed else "no")
view.style.background_gradient(subset=["circle_score", "walk_score"], cmap="YlGn").format(precision=1)

In [ ]:
# Side-by-side bar: same formula, two definitions of "within 300 m"
ax = scored.set_index("site_id")[["circle_score", "walk_score"]].plot.bar(
    figsize=(6, 3.2), color=["#9fb7d4", "#2f5f9f"], rot=0, width=0.7)
ax.set_ylabel("composite score"); ax.set_ylim(0, 100)
ax.set_title("Same sites, same formula — circle vs walking network")
ax.legend(["300 m circle", "300 m walk"], frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 7 · Interactive map

Toggle layers in the top-right. Look for places where the dashed circle covers ground the blue walk-shed never reaches: slopes, walls, the campus edge, highways. Click any point to see its OSM tags.

In [ ]:
wgs = lambda g: g.to_crs("EPSG:4326")
# Basemaps: Esri tiles need no API key (CARTO's "cartodbpositron" now shows "API KEY REQUIRED").
ESRI = "https://server.arcgisonline.com/ArcGIS/rest/services/{}/MapServer/tile/{{z}}/{{y}}/{{x}}"
m = folium.Map(location=CENTER, zoom_start=17, tiles=None, control_scale=True, max_zoom=19)
folium.TileLayer(ESRI.format("Canvas/World_Light_Gray_Base"), attr="Esri, HERE, Garmin, © OpenStreetMap contributors",
                 name="light grey", max_native_zoom=16, max_zoom=19).add_to(m)
folium.TileLayer(ESRI.format("World_Imagery"), attr="Esri, Maxar, Earthstar Geographics",
                 name="satellite", max_native_zoom=19, max_zoom=19, show=False).add_to(m)
def add(gdf, name, style=None, marker=None, show=True):
    fg = folium.FeatureGroup(name=name, show=show)
    if len(gdf) == 0:
        fg.add_to(m); return
    tip = [c for c in ["name", "shop", "highway", "railway", "leisure", "site_id"] if c in gdf.columns][:3]
    g = wgs(gdf[tip + ["geometry"]]).astype({c: str for c in tip})
    kw = {"tooltip": folium.GeoJsonTooltip(tip)} if tip else {}
    if marker:
        kw["marker"] = folium.CircleMarker(**marker)
    folium.GeoJson(g, style_function=(lambda _: style) if style else None, **kw).add_to(fg)
    fg.add_to(m)

add(L["buildings"], "buildings", {"color": "#999", "weight": 0.3, "fillColor": "#bbb", "fillOpacity": 0.25}, show=False)
add(L["green"], "green / open space", {"color": "#2f7d4f", "weight": 0.5, "fillColor": "#60a86b", "fillOpacity": 0.4})
add(gpd.GeoDataFrame(geometry=[road_zone], crs=CRS_WORK) if road_zone is not None else L["roads"],
    "major-road noise proxy (25 m)", {"color": "#9c3f34", "weight": 0, "fillColor": "#d46b5c", "fillOpacity": 0.25})
add(isos, "300 m walk-shed", {"color": "#2f5f9f", "weight": 1, "fillColor": "#2f5f9f", "fillOpacity": 0.25})
circles = gpd.GeoDataFrame(sites[["site_id"]], geometry=sites.centroid.buffer(WALK_RADIUS_M), crs=CRS_WORK)
add(circles, "300 m circle", {"color": "#333", "weight": 1.2, "dashArray": "6 4", "fillOpacity": 0})
add(L["transit"].assign(geometry=L["transit"].representative_point()), "transit",
    marker={"radius": 5, "color": "#2f5f9f", "fill": True, "fill_opacity": 1})
add(L["grocery"].assign(geometry=L["grocery"].representative_point()), "grocery",
    marker={"radius": 4, "color": "#9a641f", "fill": True, "fill_opacity": 1})
add(sites, "candidate sites", {"color": "#111", "weight": 2, "fillColor": "#f2c14e", "fillOpacity": 0.9})
for _, s in wgs(sites).iterrows():
    c = s.geometry.centroid
    folium.Marker([c.y, c.x], icon=folium.DivIcon(html=f"<b style='font-size:14px;background:#fff;padding:1px 4px;border:1px solid #222'>{s.site_id}</b>")).add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(OUT / "gis_live_osm_map.html")
m

## 8 · Static figure + exports (for slides and Grasshopper)

- `outputs/gis_live_osm_map.png`: a static map for slides or a `.tool-panel`
- `outputs/gis_live_osm_map.html`: the interactive map above
- `outputs/scored_sites_live.csv`: match on `site_id` in Grasshopper, the same as the existing `scored_sites.csv`
- `outputs/walksheds_live.geojson`: walk-shed polygons in WGS84, which you can import into Rhino via a GIS plugin

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
xmin, ymin, xmax, ymax = circles.total_bounds; pad = 120
L["buildings"].plot(ax=ax, color="#d9d6cf", linewidth=0)
if len(L["green"]): L["green"].plot(ax=ax, color="#8cc39a", alpha=0.7, linewidth=0)
if road_zone is not None: gpd.GeoSeries([road_zone], crs=CRS_WORK).plot(ax=ax, color="#d46b5c", alpha=0.25)
edges.plot(ax=ax, color="#9aa3ad", linewidth=0.4)
isos.plot(ax=ax, color="#2f5f9f", alpha=0.28, linewidth=0)
circles.boundary.plot(ax=ax, color="#222", linestyle="--", linewidth=1)
L["transit"].representative_point().plot(ax=ax, color="#2f5f9f", marker="^", markersize=40, zorder=5)
L["grocery"].representative_point().plot(ax=ax, color="#9a641f", marker="s", markersize=22, zorder=5)
sites.plot(ax=ax, color="#f2c14e", edgecolor="#111", linewidth=1.2, zorder=6)
for _, r in sites.iterrows():
    row = scored.set_index("site_id").loc[r.site_id]
    ax.annotate(f"{r.site_id}  circle {row.circle_score:.0f} → walk {row.walk_score:.0f}", (r.geometry.centroid.x, r.geometry.centroid.y),
                xytext=(8, 8), textcoords="offset points", fontsize=9, weight="bold",
                bbox={"boxstyle": "round,pad=0.2", "fc": "white", "ec": "#222", "lw": 0.6}, zorder=7)
ax.set_xlim(xmin - pad, xmax + pad); ax.set_ylim(ymin - pad, ymax + pad); ax.set_axis_off()
ax.set_title(f"300 m circle (dashed) vs 300 m walk-shed (blue)\n{SNAPSHOT} · © OpenStreetMap contributors · {CRS_WORK}", fontsize=10)
fig.tight_layout(); fig.savefig(OUT / "gis_live_osm_map.png", dpi=180); plt.show()

scored.to_csv(OUT / "scored_sites_live.csv", index=False)
isos.merge(scored[["site_id", "walk_score", "walkable_share_of_circle_pct"]]).to_crs("EPSG:4326").to_file(
    OUT / "walksheds_live.geojson", driver="GeoJSON")
print("wrote:", *[p.name for p in sorted(OUT.glob("*live*"))], sep="\n  ")

## 9 · Try it on your own site (about 5 minutes)

1. In cell 2, set `CENTER` to your studio site and add `site_square("D", lat, lon, "my site")`. Keep `DIST_M` at 1000 or less so the download stays fast.
2. `Runtime → Run all`.
3. Post the PNG and the two rank columns to Slack.

Places where the circle and the walk-shed tend to disagree: Mid-Levels, Sai Ying Pun steps, Kwun Tong podiums, anywhere a highway cuts through.

## 10 · Four-step reconstruction (for the round table)

| step | prompt for this notebook |
|---|---|
| **Did** | Which site came out on top, and did the circle and the walk-shed agree? |
| **Assumed** | Which OSM tag lists stand in for "transit", "grocery", "green" and "noise"? Are 300 m and the weights the right values for *your* user? Does walking distance treat 300 m of steps the same as 300 m on the flat? |
| **Risk** | What is missing from OSM here (mall interiors, footbridges, minibus stops)? Which of your conclusions depend on that gap? |
| **Verify** | Walk one route from the table and time it. Check one "missing" shop in Street View. Re-run with `WALK_RADIUS_M = 400`: does the ranking hold? |

*Data: © OpenStreetMap contributors, available under the Open Database License. Scores are teaching proxies, not planning evidence.*